In [30]:
import cvxpy as cp
import numpy as np
import pandas as pd
import yfinance as yf

In [31]:
n = 8
tril_idx = np.tril_indices(n)

def vech_to_sym(row):
    M = np.zeros((n, n))
    M[tril_idx] = row
    return M + M.T - np.diag(np.diag(M))

def load_rc(filename):
    df = pd.read_csv(filename, sep='\t', header=None)
    df.columns = range(df.shape[1])
    return np.stack([vech_to_sym(df.iloc[t].values) for t in range(len(df))])

# load predictions
pred_chol = load_rc("data/pred_chol_sub.txt")
pred_log  = load_rc("data/pred_log_sub.txt")
pred_dcc    = load_rc("data/pred_dcc_sub.txt")
pred_dcc_re = load_rc("data/pred_dcc_real_sub.txt")
pred_lstm   = load_rc("data/pred_lstm_rc_sub.txt")
pred_gru   = load_rc("data/pred_gru_rc_sub.txt")
pred_lstm_l   = load_rc("data/pred_lstm_logexp_rc_sub.txt")
pred_gru_l   = load_rc("data/pred_gru_logexp_rc_sub.txt")
pred_har_chol_ker = load_rc("data/pred_chol_kernel.txt")
pred_har_log_ker = load_rc("data/pred_log_kernel.txt")
pred_dcc_re_ker = load_rc("data/pred_dcc_real_kernel.txt")
pred_lstm_ker   = load_rc("data/pred_lstm_rc_kernel.txt")
pred_gru_ker   = load_rc("data/pred_gru_rc_kernel.txt")
pred_lstm_l_ker   = load_rc("data/pred_lstm_logexp_rc_kernel.txt")
pred_gru_l_ker   = load_rc("data/pred_gru_logexp_rc_kernel.txt")
n_test = len(pred_chol)

# load returns and align to test period
daily_returns = pd.read_csv("data/daily_returns.txt", sep='\t', index_col=0, parse_dates=True)
returns = daily_returns.values[-n_test:]

predictions = {
    "HAR-Cholesky": pred_chol,
    "HAR-LogExp":   pred_log,
    "DCC-st": pred_dcc,
    "DCC-real": pred_dcc_re,
    "LSTM-Cholesky": pred_lstm,
    "GRU-Cholesky": pred_gru,
    "LSTM-LogExp": pred_lstm_l,
    "GRU-LogExp": pred_gru_l,
    "HAR-Cholesky_ker": pred_har_chol_ker,
    "HAR-LogExp_ker": pred_har_log_ker,
    "DCC-real_ker": pred_dcc_re_ker,
    "LSTM-Cholesky_ker": pred_lstm_ker,
    "GRU-Cholesky_ker": pred_gru_ker,
    "LSTM-LogExp_ker": pred_lstm_l_ker,
    "GRU-LogExp_ker": pred_gru_l_ker,
}

In [32]:
def min_variance_portfolio(H):
    H = np.asarray(H, dtype=np.float64)
    H = 0.5 * (H + H.T)
    H = H + 1e-8 * (np.trace(H) / H.shape[0]) * np.eye(H.shape[0])

    n = H.shape[0]
    w = cp.Variable(n)
    objective = cp.Minimize(cp.quad_form(w, cp.psd_wrap(H)))
    constraints = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return w.value

def gmv_portfolio(H):
    n = H.shape[0]
    ones = np.ones(n)
    H_inv = np.linalg.inv(H)
    w = H_inv @ ones / (ones @ H_inv @ ones)
    return w

In [33]:
def compute_max_weight(weights):
    return np.max(weights, axis=1).mean()


def compute_effective_n_bets(weights):
    enb_per_t = 1.0 / (weights ** 2).sum(axis=1)
    return enb_per_t.mean()

In [34]:
def backtest(predictions, returns, rebalance_freq):
    T, n = returns.shape
    weights = np.zeros((T, n))
    w = np.ones(n) / n

    for t in range(T):
        if t % rebalance_freq == 0:
            w = gmv_portfolio(predictions[t])
        weights[t] = w

    portfolio_returns = (weights * returns).sum(axis=1)
    return portfolio_returns, weights

In [35]:
results_bt = {}
for name, pred in predictions.items():
    for freq_name, freq in [("daily", 1), ("weekly", 5), ("monthly", 22)]:
        port_returns, weights = backtest(pred, returns, freq)
        results_bt[f"{name} ({freq_name})"] = {
            "returns": port_returns,
            "weights": weights
        }

In [36]:
def compute_metrics(port_returns, rf=0.0):
    T = len(port_returns)
    
    ann_return = (np.prod(1 + port_returns) ** (252 / T)) - 1
    
    ann_var = np.mean(port_returns ** 2) * 252
    ann_vol = np.sqrt(ann_var)
    
    sharpe = (ann_return - rf) / ann_vol

    cum_returns = np.cumprod(1 + port_returns)
    peak = np.maximum.accumulate(cum_returns)
    drawdown = (peak - cum_returns) / peak
    mdd = drawdown.max()
    
    return {
        "Ann. Return":  round(ann_return, 4),
        "Ann. Vol":     round(ann_vol, 4),
        "Sharpe":       round(sharpe, 4),
        "MDD":          round(mdd, 4),
    }
def compute_turnover(weights, rebalance_freq):
    T = len(weights)
    turnovers = []
    for t in range(1, T):
        if t % rebalance_freq == 0:
            turnover = np.sum(np.abs(weights[t] - weights[t-1]))
            turnovers.append(turnover)
    return np.array(turnovers)

def compute_breakeven_cost(port_returns, weights, rebalance_freq):
    turnovers = compute_turnover(weights, rebalance_freq)
    avg_return = port_returns.mean()
    avg_turnover = turnovers.mean()
    C = rebalance_freq * avg_return / avg_turnover
    return C, avg_turnover

In [37]:
freq_map = {"daily": 1, "weekly": 5, "monthly": 22}

metrics = {}
for name, res in results_bt.items():
    freq_name = name.split("(")[1].replace(")", "").strip()
    freq = freq_map[freq_name]
    
    C, avg_turnover = compute_breakeven_cost(
        res["returns"], res["weights"], freq
    )
    
    m = compute_metrics(res["returns"])
    m["Avg Turnover"] = round(avg_turnover, 4)
    m["Break-even C"] = round(C, 6)

    m["Max Weight"]     = round(compute_max_weight(res["weights"]), 4) 
    m["Ef n bets"]  = round(compute_effective_n_bets(res["weights"]), 4)   
    metrics[name] = m

pd.DataFrame(metrics).T

,Ann. Return,Ann. Vol,Sharpe,MDD,Avg Turnover,Break-even C,Max Weight,Ef n bets
HAR-Cholesky (daily),0.0655,0.1185,0.5532,0.1408,0.5275,0.000530,0.6682,1.8930
HAR-Cholesky (weekly),0.0392,0.1177,0.3334,0.1479,0.7155,0.001260,0.6565,1.9451
HAR-Cholesky (monthly),0.0302,0.1202,0.2511,0.1672,0.9546,0.003381,0.7020,1.8205
HAR-LogExp (daily),0.0745,0.1156,0.6451,0.1355,0.3226,0.000966,0.6227,2.2656
HAR-LogExp (weekly),0.0624,0.1150,0.5424,0.1485,0.4884,0.002726,0.6243,2.2896
HAR-LogExp (monthly),0.0431,0.1173,0.3671,0.1495,0.6468,0.006622,0.6341,2.1471
DCC-st (daily),0.0921,0.1199,0.7683,0.1293,0.3984,0.000949,0.7114,1.6185
DCC-st (weekly),0.0671,0.1226,0.5478,0.1324,0.9467,0.001519,0.7162,1.5867
DCC-st (monthly),0.0680,0.1201,0.5658,0.1375,1.3422,0.004746,0.7060,1.6322
DCC-real (daily),0.1405,0.1250,1.1238,0.1376,0.3988,0.001386,0.6832,1.2236


In [38]:

final_dates = pd.read_csv("data/final_dates.csv")["date"]
final_dates = pd.to_datetime(final_dates)
test_dates = final_dates.iloc[-600:].dt.normalize()
start_dt = (test_dates.min() - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
end_dt   = (test_dates.max() + pd.Timedelta(days=2)).strftime('%Y-%m-%d')

sp500 = yf.download("^GSPC", start=start_dt, end=end_dt,
                    auto_adjust=True, progress=False)
prices = sp500["Close"].squeeze()
sp_returns = prices.pct_change().dropna()
sp_returns.index = sp_returns.index.normalize()

test_set = set(test_dates)
sp_returns_aligned = sp_returns[sp_returns.index.isin(test_set)]

print(f"Test dates from final_dates.csv: {len(test_dates)}")
print(f"S&P returns aligned:              {len(sp_returns_aligned)}")
print(f"Date range: {sp_returns_aligned.index[0].date()} to "
      f"{sp_returns_aligned.index[-1].date()}")

missing_in_sp = test_set - set(sp_returns_aligned.index)
if missing_in_sp:
    print(f"\nWARNING: {len(missing_in_sp)} test dates not found in S&P data.")
    print(f"  Examples: {sorted(missing_in_sp)[:5]}")
else:
    print("\nAll test dates matched.")

sp_metrics = compute_metrics(sp_returns_aligned.values, rf=0.0)
print("\nS&P 500 metrics (aligned to test dates):", sp_metrics)

Test dates from final_dates.csv: 600
S&P returns aligned:              600
Date range: 2022-08-03 to 2024-12-30

All test dates matched.

S&P 500 metrics (aligned to test dates): {'Ann. Return': np.float64(0.1555), 'Ann. Vol': np.float64(0.1545), 'Sharpe': np.float64(1.0065), 'MDD': np.float64(0.1691)}


In [39]:
n_assets = returns.shape[1]
n_test = 600

ew_weights = np.full((n_test, n_assets), 1.0 / n_assets)

ew_returns = returns.mean(axis=1)
ew_returns = ew_returns[~np.isnan(ew_returns)]
ew_returns = ew_returns[-n_test:]

print(f"Number of returns: {len(ew_returns)}")

ew_metrics = compute_metrics(ew_returns, rf=0.0)

ew_metrics["Max Weight"] = round(float(np.max(ew_weights, axis=1).mean()), 4)
ew_metrics["Ef n bets"] = round(float(compute_effective_n_bets(ew_weights)), 4)

ew_metrics["Avg Turnover"] = 0.0
ew_metrics["Break-even C"] = float('inf')  

print("Equal-weight (1/N):", ew_metrics)

Number of returns: 600
Equal-weight (1/N): {'Ann. Return': np.float64(0.1117), 'Ann. Vol': np.float64(0.143), 'Sharpe': np.float64(0.7816), 'MDD': np.float64(0.1541), 'Max Weight': 0.125, 'Ef n bets': 8.0, 'Avg Turnover': 0.0, 'Break-even C': inf}
